<a href="https://colab.research.google.com/github/laurafarage/data-analyst-portfolio/blob/main/marketing_etl_automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
from datetime import date
import gspread
import gspread_dataframe as gd
from gspread_dataframe import set_with_dataframe
from oauth2client.service_account import ServiceAccountCredentials
!pip install oauth2client

from google.colab import drive
drive.mount('/content/drive')

scope = [""]
creds = ServiceAccountCredentials.from_json_keyfile_name('/content/drive/MyDrive/automação/settings .json', scope)
gc = gspread.authorize(creds)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Inserir Link Planilha
link = 'https://docs.google.com/spreadsheets/d/1P_GgBWDcAeh_7phif1rmA24BIOibEW4s5jvpoK0aMbI/edit?usp=sharing'

In [ ]:
# Inserir Planilha Meta
worksheet_meta = gc.open_by_url(link).worksheet('Meta')

# Obter os dados da planilha em um DataFrame
dataframe_meta = pd.DataFrame(worksheet_meta.get_all_records())

# Renomear as colunas do DataFrame
dataframe_meta.rename(columns={

   'Date': 'data',
   'Campaign name': 'Campaign name',
   'Impressions': 'Impressões',
   'Cost': 'Custo',
   'Reach': 'Alcance',
   'Frequency': 'Frequencia',
   'Post engagements': 'Engajamento',
   'Video watches at 25%': 'Video views 25%',
   'Video watches at 50%': 'Video views 50%',
   'Video watches at 75%': 'Video views 75%',
   'Video watches at 100%': 'Video views 100%',
   'Video play actions': 'Video views',
   'Link clicks': 'Link clicks',
   'Post engagements': 'Engagements',

}, inplace=True)

dataframe_meta = dataframe_meta.reset_index(drop=True)




#Adicionar a coluna Plataforma_geral e definir seu valor baseado na coluna Publisher Platform
dataframe_meta['Plataforma_geral'] = dataframe_meta['Publisher platform']
dataframe_meta['Plataforma_pacing'] = 'Meta'

def definir_tema(nome_campanha):
    nome_campanha = nome_campanha.upper()
    if 'SEGURANÇA' in nome_campanha:
        return 'SEGURANÇA DIGITAL'
    elif 'ESG' in nome_campanha:
        return 'ESG'
    elif 'EDUCAÇÃO FINANCEIRA' in nome_campanha:
        return 'EDUCAÇÃO FINANCEIRA'
    elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
        return 'CULTURA E ESPORTE'
    elif 'BENEFÍCIOS SOCIAIS' in nome_campanha:
        return 'BENEFÍCIOS SOCIAIS'
    elif 'OUTROS' in nome_campanha:
        return 'OUTROS'
    else:
        return 'OUTROS'

# Aplicar a função para definir o tema na nova coluna
dataframe_meta['TEMA'] = dataframe_meta['Campaign name'].apply(definir_tema)

# Função para definir o tipo de compra com base no conteúdo da coluna "Campaign name"
def definir_tipo_de_compra(nome_campanha):
    if 'CPE' in nome_campanha.upper():
        return 'CPE'
    elif 'CPM' in nome_campanha.upper():
        return 'CPM'
    elif 'CPV' in nome_campanha.upper():
        return 'CPV'
    elif 'CPC' in nome_campanha.upper():
        return 'CPC'
    elif 'SEARCH' in nome_campanha.upper():
        return 'SEARCH'
    else:
        return 'Outro'

# Aplicar a função para definir o tipo de compra na nova coluna
dataframe_meta['Tipo de Compra'] = dataframe_meta['Campaign name'].apply(definir_tipo_de_compra)



In [ ]:
# # Inserir Planilha Pinterest
# worksheet_pinterest = gc.open_by_url(link).worksheet('Pinterest')

# # Obter os dados da planilha em um DataFrame
# dataframe_pinterest = pd.DataFrame(worksheet_pinterest.get_all_records())

# # Renomear as colunas do DataFrame
# dataframe_pinterest.rename(columns={

#    'Date': 'data',
#    'Campaign name': 'Campaign name',
#    'Impressions': 'Impressões',
#    'Spend in account currency': 'Custo',
#    'Reach': 'Alcance',
#    'Frequency': 'Frequencia',
#    'Post engagements': 'Engajamento',
#    'Total video played at 25%': 'Video views 25%',
#    'Total video played at 50%': 'Video views 50%',
#    'Total video played at 75%': 'Video views 75%',
#    'Total video played at 100%': 'Video views 100%',
#    'Video views': 'Video views',
#    'Pin clicks': 'Link clicks',
#    'Post engagements': 'Engagements',
#    'Ad set name': 'Ad name',

# }, inplace=True)

# dataframe_pinterest = dataframe_pinterest.reset_index(drop=True)

# #Adicionar a coluna Plataforma_geral e definir seu valor baseado na coluna Publisher Platform
# dataframe_pinterest['Plataforma_geral'] = 'Pinterest'
# dataframe_pinterest['Plataforma_pacing'] = 'Pinterest'

# def definir_tema(nome_campanha):
#     nome_campanha = nome_campanha.upper()
#     if 'SEGURANÇA' in nome_campanha:
#         return 'SEGURANÇA DIGITAL'
#     elif 'ESG' in nome_campanha:
#         return 'ESG'
#     elif 'EDUCAÇÃO FINANCEIRA' in nome_campanha:
#         return 'EDUCAÇÃO FINANCEIRA'
#     elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
#         return 'CULTURA E ESPORTE'
#     elif 'BENEFÍCIOS SOCIAIS' in nome_campanha:
#         return 'BENEFÍCIOS SOCIAIS'
#     elif 'OUTROS' in nome_campanha:
#         return 'OUTROS'
#     else:
#         return 'INDEFINIDO'

# # Aplicar a função para definir o tema na nova coluna
# dataframe_pinterest['TEMA'] = dataframe_pinterest['Campaign name'].apply(definir_tema)

# # Função para definir o tipo de compra com base no conteúdo da coluna "Campaign name"
# def definir_tipo_de_compra(nome_campanha):
#     if 'CPE' in nome_campanha.upper():
#         return 'CPE'
#     elif 'CPM' in nome_campanha.upper():
#         return 'CPM'
#     elif 'CPV' in nome_campanha.upper():
#         return 'CPV'
#     elif 'SEARCH' in nome_campanha.upper():
#         return 'SEARCH'
#     else:
#         return 'Outro'

# # Aplicar a função para definir o tipo de compra na nova coluna
# dataframe_pinterest['Tipo de Compra'] = dataframe_pinterest['Campaign name'].apply(definir_tipo_de_compra)

In [ ]:
#Inserir Planilha TikTok
worksheet_tiktok = gc.open_by_url(link).worksheet('TikTok')

# Obter os dados da planilha em um DataFrame
dataframe_tiktok = pd.DataFrame(worksheet_tiktok.get_all_records())

# Renomear as colunas do DataFrame
dataframe_tiktok.rename(columns={

    'Date': 'data',
    'Campaign name': 'Campaign name',
    'Impressions': 'Impressões',
    'Cost': 'Custo',
    'Reach': 'Alcance',
    'Frequency': 'Frequencia',
    'Post engagements': 'Engajamento',
    'Video views at 25%': 'Video views 25%',
    'Video views at 50%': 'Video views 50%',
    'Video views at 75%': 'Video views 75%',
    'Video views at 100%': 'Video views 100%',
    'Video views': 'Video views',
    'Link clicks': 'Link clicks',
    'Post engagements': 'Engagements',
    'Clicks': 'Link clicks',
    'Ad name': 'Ad name',
    '6-second video views (focused view)': 'Views focused',
    'Video thumbnail URL': 'Destination URL',

 }, inplace=True)

dataframe_tiktok = dataframe_tiktok.reset_index(drop=True)

 # Criar a coluna "Ad set name" com o mesmo valor de "Ad name"
dataframe_tiktok['Ad set name'] = dataframe_tiktok['Ad name']

#Adicionar a coluna Plataforma_geral e definir seu valor baseado na coluna Publisher Platform

dataframe_tiktok['Plataforma_geral'] = 'TikTok'
dataframe_tiktok['Plataforma_pacing'] = 'TikTok'

def definir_tema(nome_campanha):
    nome_campanha = nome_campanha.upper()
    if 'SEGURANÇA' in nome_campanha:
        return 'SEGURANÇA DIGITAL'
    elif 'ESG' in nome_campanha:
        return 'ESG'
    elif 'EDUCAÇÃO FINANCEIRA' in nome_campanha:
        return 'EDUCAÇÃO FINANCEIRA'
    elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
        return 'CULTURA E ESPORTE'
    elif 'BENEFÍCIOS SOCIAIS' in nome_campanha:
        return 'BENEFÍCIOS SOCIAIS'
    elif 'OUTROS' in nome_campanha:
        return 'OUTROS'
    else:
        return 'INDEFINIDO'

# Aplicar a função para definir o tema na nova coluna
dataframe_tiktok['TEMA'] = dataframe_tiktok['Campaign name'].apply(definir_tema)

# Função para definir o tipo de compra com base no conteúdo da coluna "Campaign name"
def definir_tipo_de_compra(nome_campanha):
     if 'CPE' in nome_campanha.upper():
         return 'CPV 6seg'
     elif 'CPM' in nome_campanha.upper():
         return 'CPM'
     elif 'CPV' in nome_campanha.upper():
         return 'CPV'
     elif 'CPC' in nome_campanha.upper():
         return 'CPC'
     elif 'SEARCH' in nome_campanha.upper():
         return 'SEARCH'
     else:
         return 'Outro'

 # Aplicar a função para definir o tipo de compra na nova coluna
dataframe_tiktok['Tipo de Compra'] = dataframe_tiktok['Campaign name'].apply(definir_tipo_de_compra)

In [ ]:
#   #Inserir Planilha Linkedin
# worksheet_linkedin = gc.open_by_url(link).worksheet('LinkedIn')

# # Obter os dados da planilha em um DataFrame
# dataframe_linkedin = pd.DataFrame(worksheet_linkedin.get_all_records())

# # Renomear as colunas do DataFrame
# dataframe_linkedin.rename(columns={

#   'Date': 'data',
#    'Campaign name': 'Campaign name',
#   'Impressions': 'Impressões',
#     'Total spent': 'Custo',
#     'Reactions': 'Post reactions',
#     'Comments': 'Post comments',
#     'Shares': 'Post shares',
#     'Total engagements': 'Engajamento',
#     'Video views at 25%': 'Video views 25%',
#     'Video views at 50%': 'Video views 50%',
#     'Video views at 75%': 'Video views 75%',
#     'Video views at 100%': 'Video views 100%',
#     'Video views ': 'Video views',
#     'Link clicks': 'Link clicks',
#     'Clicks': 'Link clicks',
#     'Creative title': 'Ad name',

#  }, inplace=True)

# dataframe_linkedin = dataframe_linkedin.reset_index(drop=True)

# # # Criar a coluna "Ad set name" com o mesmo valor de "Ad name"
# dataframe_linkedin['Ad set name'] = dataframe_linkedin['Ad name']

# # #Adicionar a coluna Plataforma_geral e definir seu valor baseado na coluna Publisher Platform

# dataframe_linkedin['Plataforma_geral'] = 'Linkedin'
# dataframe_linkedin['Plataforma_pacing'] = 'Linkedin'

# def definir_tema(nome_campanha):
#     nome_campanha = nome_campanha.upper()
#     if 'SEGURANÇA' in nome_campanha:
#         return 'SEGURANÇA DIGITAL'
#     elif 'ESG' in nome_campanha:
#         return 'ESG'
#     elif 'EDUCAÇÃO FINANCEIRA' in nome_campanha:
#         return 'EDUCAÇÃO FINANCEIRA'
#     elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
#         return 'CULTURA E ESPORTE'
#     elif 'BENEFÍCIOS SOCIAIS' in nome_campanha:
#         return 'BENEFÍCIOS SOCIAIS'
#     elif 'OUTROS' in nome_campanha:
#         return 'OUTROS'
#     else:
#         return 'INDEFINIDO'

# # Aplicar a função para definir o tema na nova coluna
# dataframe_linkedin['TEMA'] = dataframe_linkedin['Campaign name'].apply(definir_tema)

# # # Função para definir o tipo de compra com base no conteúdo da coluna "Campaign name"
# def definir_tipo_de_compra(nome_campanha):
#     if 'CPE' in nome_campanha.upper():
#         return 'CPE'
#     elif 'CPM' in nome_campanha.upper():
#         return 'CPM'
#     elif 'CPV' in nome_campanha.upper():
#         return 'CPV'
#     elif 'CPC' in nome_campanha.upper():
#         return 'CPC'
#     elif 'SEARCH' in nome_campanha.upper():
#         return 'SEARCH'
#     else:
#         return 'Outro'

# # # Aplicar a função para definir o tipo de compra na nova coluna
# dataframe_linkedin['Tipo de Compra'] = dataframe_linkedin['Campaign name'].apply(definir_tipo_de_compra)

In [ ]:
worksheet_kwai = gc.open_by_url(link).worksheet('Kwai_ID60160196')

#Obter os dados da planilha em um DataFrame
dataframe_kwai = pd.DataFrame(worksheet_kwai.get_all_records())

#Renomear as colunas do DataFrame
dataframe_kwai.rename(columns={

   'Time': 'data',
   'Campaign name': 'Campaign name',
   'Impression': 'Impressões',
   'Cost(BRL)': 'Custo',
   'Reach (UTC+00:00)': 'Alcance',
   'Frequency': 'Frequencia',
   'Post engagements': 'Engajamento',
   'Video plays at 25%': 'Video views 25%',
   'Video plays at 50%': 'Video views 50%',
   'Video plays at 75%': 'Video views 75%',
   'Video completions': 'Video views 100%',
   '3s Video Plays': 'Video views',
   'Clicks': 'Link clicks',
   'Creative name': 'Ad set name',
   'Click': 'Link clicks'

  }, inplace=True)

# Criar a coluna "Ad set name" com o mesmo valor de "Ad name"
dataframe_kwai['Ad name'] = dataframe_kwai['Ad set name']


dataframe_kwai = dataframe_kwai.reset_index(drop=True)

dataframe_kwai['Plataforma_geral'] = 'Kwai'
dataframe_kwai['Plataforma_pacing'] = 'Kwai'

def definir_tema(nome_campanha):
    nome_campanha = nome_campanha.upper()
    if 'SEGURANÇA' in nome_campanha:
        return 'SEGURANÇA DIGITAL'
    elif 'ESG' in nome_campanha:
        return 'ESG'
    elif 'EDUCAÇÃO FINANCEIRA' in nome_campanha:
        return 'EDUCAÇÃO FINANCEIRA'
    elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
        return 'CULTURA E ESPORTE'
    elif 'BENEFÍCIOS SOCIAIS' in nome_campanha:
        return 'BENEFÍCIOS SOCIAIS'
    elif 'OUTROS' in nome_campanha:
        return 'OUTROS'
    elif 'E. FINACEIRA' in nome_campanha:
        return 'EDUCAÇÃO FINANCEIRA'
    else:
        return 'INDEFINIDO'

# Aplicar a função para definir o tema na nova coluna
dataframe_kwai['TEMA'] = dataframe_kwai['Campaign name'].apply(definir_tema)

# Função para definir o tipo de compra com base no conteúdo da coluna "Campaign name"
def definir_tipo_de_compra(nome_campanha):
     if 'CPE' in nome_campanha.upper():
         return 'CPE'
     elif 'CPM' in nome_campanha.upper():
         return 'CPM'
     elif 'CPV' in nome_campanha.upper():
         return 'CPV'
     elif 'CPC' in nome_campanha.upper():
         return 'CPC'
     else:
         return 'Outro'

# Aplicar a função para definir o tipo de compra na nova coluna
dataframe_kwai['Tipo de Compra'] = dataframe_kwai['Campaign name'].apply(definir_tipo_de_compra)


In [ ]:
# # Inserir Planilha Google
# worksheet_google = gc.open_by_url(link).worksheet('Google')
# dataframe_google = pd.DataFrame(worksheet_google.get_all_records())

# # Renomear as colunas do DataFrame
# dataframe_google.rename(columns={
#     'Date': 'data',
#     'Campaign name': 'Campaign name',
#     'Impressions': 'Impressões',
#     'Cost': 'Custo',
#     'Post engagements': 'Engajamento',
#     'Watch 25% views': 'Video views 25%',
#     'Watch 50% views': 'Video views 50%',
#     'Watch 75% views': 'Video views 75%',
#     'Watch 100% views': 'Video views 100%',
#     'Video views': 'Video views',
#     'Clicks': 'Link clicks',
#     'Headline': 'Ad name',
#     'Final URL': 'Destination URL',
# }, inplace=True)

# # Reset e plataformas
# dataframe_google = dataframe_google.reset_index(drop=True)
# dataframe_google['Plataforma_geral'] = 'Google'
# dataframe_google['Plataforma_pacing'] = 'Google'

# # Tema
# def definir_tema(nome_campanha):
#     nome_campanha = nome_campanha.upper()
#     if 'SEGURANÇA' in nome_campanha:
#         return 'SEGURANÇA DIGITAL'
#     elif 'ESG' in nome_campanha:
#         return 'ESG'
#     elif 'EDUCAÇÃO FINANCEIRA' in nome_campanha or 'E. FINACEIRA' in nome_campanha:
#         return 'EDUCAÇÃO FINANCEIRA'
#     elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
#         return 'CULTURA E ESPORTE'
#     elif 'BENEFÍCIOS SOCIAIS' in nome_campanha:
#         return 'BENEFÍCIOS SOCIAIS'
#     elif 'OUTROS' in nome_campanha:
#         return 'OUTROS'
#     else:
#         return 'INDEFINIDO'

# dataframe_google['TEMA'] = dataframe_google['Campaign name'].apply(definir_tema)

# # Tipo de compra
# def definir_tipo_de_compra(nome_campanha):
#     nome = nome_campanha.upper()
#     if 'CPE' in nome:
#         return 'CPE'
#     elif 'CPM' in nome:
#         return 'CPM'
#     elif 'CPV' in nome:
#         return 'CPV'
#     elif 'CPC' in nome:
#         return 'CPC'
#     elif 'YOUTUBE' in nome:
#         return 'YOUTUBE'
#     elif 'MASTERHEAD' in nome:
#         return 'MASTERHEAD'
#     elif 'SEARCH' in nome:
#         return 'SEARCH'
#     else:
#         return 'Outro'

# dataframe_google['Tipo de compra'] = dataframe_google['Campaign name'].apply(definir_tipo_de_compra)

In [ ]:
merged_dataframe = pd.concat([dataframe_meta, dataframe_tiktok, dataframe_kwai])

# Converte a coluna 'data' para datetime e cria coluna de mês
merged_dataframe['data'] = pd.to_datetime(merged_dataframe['data'], dayfirst=True, errors='coerce')
merged_dataframe['mês'] = merged_dataframe['data'].dt.month_name()

# Extrai secundagem do "Ad Name" quando "Tipo de Compra" for CPV ou CPV 6seg
def extrair_secundagem(ad_name):
    match = re.search(r'(\d+)"', str(ad_name))
    if match:
        return int(match.group(1))
    return None

condicao_cpv = merged_dataframe['Tipo de Compra'].isin(['CPV', 'CPV 6seg'])
merged_dataframe.loc[condicao_cpv, 'Segundos'] = merged_dataframe.loc[condicao_cpv, 'Ad name'].apply(extrair_secundagem)

# --- Conexão com o Google Sheets ---
# link = 'URL da sua planilha'
# gc = gspread.authorize(credenciais)

planilha = gc.open_by_url(link)

# Nome da guia onde os dados serão escritos
guia = planilha.worksheet('Planilha Unificada')

# Limpa os dados existentes (opcional)
guia.clear()

# Escreve o dataframe na planilha
set_with_dataframe(guia, merged_dataframe)




/tmp/ipykernel_2311/1763693511.py:4: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  merged_dataframe['data'] = pd.to_datetime(merged_dataframe['data'], dayfirst=True, errors='coerce')


In [ ]:
# # Código Orgânico

# #Inserir Planilha orgânicos Facebook
# worksheet_facebook_organicos = gc.open_by_url(link).worksheet('FB_Organicos')

# #Obter os dados da planilha em um DataFrame
# dataframe_facebook_organicos = pd.DataFrame(worksheet_facebook_organicos.get_all_records())

# #Renomear as colunas do DataFrame
# dataframe_facebook_organicos.rename(columns={
#    'Post creation date': 'data',
#    'Post reach': 'Alcance',
#    'Post impressions': 'Impressões',
#    'Likes on posts': 'Likes',
#    'Comments on posts': 'Comments',
#    'Shares on posts': 'Shares',
#    'Post link clicks': 'Link clicks',

#    }, inplace=True)

# dataframe_facebook_organicos['Plataforma'] = 'Facebook'

# def definir_tema(nome_campanha):
#     nome_campanha = nome_campanha.upper()
#     if '#SEGURANÇA' in nome_campanha:
#         return 'SEGURANÇA DIGITAL'
#     elif '#SUSTENTABILIDADE' in nome_campanha:
#         return 'ESG'
#     elif '#STARTUP' in nome_campanha:
#         return 'INOVAÇÃO'
#     elif '#EDUCAÇÃO' in nome_campanha:
#         return 'EDUCAÇÃO FINANCEIRA'
#     elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
#         return 'CULTURA E ESPORTE'
#     elif '#BENEFÍCIO' in nome_campanha:
#         return 'BENEFÍCIOS SOCIAIS'
#     elif '#BRANDING' in nome_campanha:
#         return 'OUTROS'
#     else:
#         return 'INDEFINIDO'

# # Aplicar a função para definir o tema na nova coluna
# dataframe_facebook_organicos['TEMA'] = dataframe_facebook_organicos['Post message'].apply(definir_tema)

# #Inserir Planilha orgânicos Instagram
# worksheet_instagram_organicos = gc.open_by_url(link).worksheet('IG_Organicos')

# #Obter os dados da planilha em um DataFrame
# dataframe_instagram_organicos = pd.DataFrame(worksheet_instagram_organicos.get_all_records())

# #Renomear as colunas do DataFrame
# dataframe_instagram_organicos.rename(columns={
#    'Date': 'data',
#    'Media caption': 'Post message',
#    'Media reach': 'Alcance',
#    'Views': 'Impressões',
#    'Like count': 'Likes',
#    'Unique saves': 'Saves',
#    'Comments count': 'Comments',
#    'Media permalink': 'Link to post'


#    }, inplace=True)

# dataframe_instagram_organicos['Plataforma'] = 'Instagram'

# def definir_tema(nome_campanha):
#     nome_campanha = nome_campanha.upper()
#     if '#SEGURANÇA' in nome_campanha:
#         return 'SEGURANÇA DIGITAL'
#     elif '#SUSTENTABILIDADE' in nome_campanha:
#         return 'ESG'
#     elif '#STARTUP' in nome_campanha:
#         return 'INOVAÇÃO'
#     elif '#EDUCAÇÃO' in nome_campanha:
#         return 'EDUCAÇÃO FINANCEIRA'
#     elif 'CULTURA' in nome_campanha or 'ESPORTE' in nome_campanha:
#         return 'CULTURA E ESPORTE'
#     elif '#BENEFÍCIO' in nome_campanha:
#         return 'BENEFÍCIOS SOCIAIS'
#     elif '#BRANDING' in nome_campanha:
#         return 'OUTROS'
#     else:
#         return 'INDEFINIDO'

# # Aplicar a função para definir o tema na nova coluna
# dataframe_instagram_organicos['TEMA'] = dataframe_instagram_organicos['Post message'].apply(definir_tema)

#    #Juntar os DataFrames
# merged_dataframe = pd.concat([dataframe_facebook_organicos,dataframe_instagram_organicos])

# planilha = gc.open_by_url(link)

# #Selecione a guia específica onde deseja salvar os dados (substitua 'Nome da Guia' pelo nome correto)
# guia = planilha.worksheet('Organicos Unificada')

# #Limpe os dados existentes na guia (opcional)
# guia.clear()

# #Escreva o DataFrame na guia
# set_with_dataframe(guia, merged_dataframe)

# print("Dados enviados com sucesso para a planilha no Google Sheets.")


### ***Código região***

In [ ]:
#Inserir Planilha Meta
worksheet_meta_regiao = gc.open_by_url(link).worksheet('Meta Regiao')

#Obter os dados da planilha em um DataFrame
dataframe_meta_regiao = pd.DataFrame(worksheet_meta_regiao.get_all_records())

#Renomear as colunas do DataFrame
dataframe_meta_regiao.rename(columns={
   'Date': 'data',
   'clicks': 'Link clicks',
   'Cost': 'Custo',
   'purchase_type_format': 'Tipo de compra',
   'Video play actions': 'Video views',
   'Video watches at 25%': 'Video views 25%',
   'Video watches at 50%': 'Video views 50%',
   'Video watches at 75%': 'Video views 75%',
   'Video watches at 100%': 'Video views 100%',
   'Impressions': 'Impressões',
   'campaign_name': 'Campaign name',
   'format_name': 'Ad name',
   'creative_redirect_url': 'URL',
   'site_name': 'Site',
   'Reach (UTC+00:00)': 'Alcance',
   'Region': 'Região',
   'frequency': 'Frequência',
   'Ad name': 'Criativo',
   'Ad creative object type': 'Objective',

}, inplace=True)


dataframe_meta_regiao['Plataforma'] = 'Meta'

In [ ]:
# Inserir Planilha Kwai
#worksheet_linkedin = gc.open_by_url(link).worksheet('Kwai Regiao')


# Supondo que a guia se chama "Pagina_Regiao"
#worksheet13 = spreadsheet.worksheet('Kwai Regiao')

# Obter os dados da planilha em um DataFrame
#dataframe_kwai_regiao = pd.DataFrame(worksheet13.get_all_records())

# Renomear as colunas do DataFrame
#dataframe_kwai_regiao.rename(columns={
#    'Time': 'data',
#    'Campaign name': 'Campaign name',
#    'Charging Method': 'Tipo de compra',
#    'Ad Set Name': 'Criativo',
#    'Cost(BRL)': 'Custo',
#    'Impression': 'Impressões',
#    'Click': 'URL Clicks',
#    '3s Video Plays': 'Video views',
#    'Counts of video played to its completion': 'Video views 100%',
#    'Third Party Impression Tracking URL': 'URL',
#    'Subregion/State': "Região",
#    'Plataforma_geral':'Publisher platform'

# }, inplace=True)

# Exibir o DataFrame resultante
#dataframe_kwai_regiao['data'] = pd.to_datetime(dataframe_kwai_regiao['data'], errors='coerce')

#dataframe_kwai_regiao.loc[dataframe_kwai_regiao['Região'] == 'Santa Catarina', 'Região'] = 'Santa Catarina - Brazil'
#dataframe_kwai_regiao['data'] = pd.to_datetime(dataframe_kwai_regiao['data'], errors='coerce')

#dataframe_kwai_regiao['Plataforma_geral'] = 'kwai'
#dataframe_kwai_regiao['Plataforma'] = 'kwai'

In [ ]:
#Juntar os DataFrames
merged_dataframe = pd.concat([dataframe_meta_regiao])

planilha = gc.open_by_url(link)

#Selecione a guia específica onde deseja salvar os dados (substitua 'Nome da Guia' pelo nome correto)
guia = planilha.worksheet('regiao unificada')

#Limpe os dados existentes na guia (opcional)
guia.clear()

#Escreva o DataFrame na guia
set_with_dataframe(guia, merged_dataframe)

print("Dados enviados com sucesso para a planilha no Google Sheets.")

Dados enviados com sucesso para a planilha no Google Sheets.


### ***Código Idade***

In [ ]:
#Inserir Planilha Meta
# Abrir a planilha pelo URL
spreadsheet = gc.open_by_url(link)

# Supondo que a guia se chama "Meta Idade genero"
worksheet20 = spreadsheet.worksheet('Meta Idade genero')

#Obter os dados da planilha em um DataFrame
dataframe_meta_idade = pd.DataFrame(worksheet20.get_all_records())


#Renomear as colunas do DataFrame
dataframe_meta_idade.rename(columns={
   'Date': 'data',
   'clicks': 'URL Clicks',
   'Cost': 'Custo',
   'purchase_type_format': 'Tipo de compra',
   'Video play actions': 'Video views',
   'Video watches at 25%': 'Video views 25%',
   'Video watches at 50%': 'Video views 50%',
   'Video watches at 75%': 'Video views 75%',
   'Video watches at 100%': 'Video views 100%',
   'Impressions': 'Impressões',
   'campaign_name': 'Campaign name',
   'format_name': 'Ad name',
   'creative_redirect_url': 'URL',
   'site_name': 'Site',
   'Reach (UTC+00:00)': 'Alcance',
   'frequency': 'Frequência',
   'Age': 'Idade',

}, inplace=True)

dataframe_meta_idade['Plataforma'] = 'Meta'
dataframe_meta_idade['Plataforma_geral'] = 'Meta'

In [ ]:
# # Abrir a planilha pelo URL
# spreadsheet = gc.open_by_url(link)

# #Supondo que a guia se chama "Pagina_Regiao"
# worksheet21 = spreadsheet.worksheet('Kwai Idade')

# #Obter os dados da planilha em um DataFrame
# dataframe_kwai_idade = pd.DataFrame(worksheet21.get_all_records())

# #Renomear as colunas do DataFrame
# dataframe_kwai_idade.rename(columns={
#    'Time': 'data',
#    'Ad Set Name': 'Ad name',
#    'Charging Method': 'Tipo de compra',
#    'Creative name': 'Criativo',
#    'Cost(BRL)': 'Custo',
#     'Impression': 'Impressões',
#    '3s Video Plays': 'Video views',
#    'Counts of video played to its completion': 'Video views 100%',
#    'Third Party Impression Tracking URL': 'URL',
#    'Age': 'Idade',
#     'Click': 'Link clicks',

# }, inplace=True)


# dataframe_kwai_idade['Plataforma_geral'] = 'kwai'
# dataframe_kwai_idade['Plataforma'] = 'kwai'

In [ ]:
#Juntar os DataFrames
merged_dataframe = pd.concat([dataframe_meta_idade])

#Mapear e substituir as informações da coluna 'Gender'
gender_mapping = {'female': 'Female', 'male': 'Male', 'unknown': 'Unknown'}
merged_dataframe['Gender'] = merged_dataframe['Gender'].str.lower().map(gender_mapping).fillna(merged_dataframe['Gender'])

#Abra a planilha específica pelo URL
planilha = gc.open_by_url(link)

#Selecione a guia específica onde deseja salvar os dados (substitua 'Nome da Guia' pelo nome correto)
guia = planilha.worksheet('Idade Unificada')

#Limpe os dados existentes na guia (opcional)
guia.clear()

#Escreva o DataFrame na guia
set_with_dataframe(guia, merged_dataframe)

Contatenar Adserver

In [ ]:
# Inserir Planilha adserver_julho
#link30 = 'https://docs.google.com/spreadsheets/d/1WuD3XWtOkH4vQ8ufrUBgIArECBQlAUmaNJ1C2FMWl-U/edit?usp=sharing'
#spreadsheet = gc.open_by_url(link30)

# Supondo que a guia se chama "Pagina_Regiao"
#worksheet30 = spreadsheet.worksheet('Adserver_julho')

# Obter os dados da planilha em um DataFrame
#dataframe_adserverjulho = pd.DataFrame(worksheet30.get_all_records())

#dataframe_adserverjulho['Plataforma'] = 'Julho'
#dataframe_adserverjulho['Plataforma_geral'] = 'Julho'

In [ ]:
# Inserir Planilha adserver_agosto
#link31 = 'https://docs.google.com/spreadsheets/d/1WuD3XWtOkH4vQ8ufrUBgIArECBQlAUmaNJ1C2FMWl-U/edit?usp=sharing'
#spreadsheet = gc.open_by_url(link31)

# Supondo que a guia se chama "Pagina_Regiao"
#worksheet31 = spreadsheet.worksheet('Adserver_agosto')

# Obter os dados da planilha em um DataFrame
#dataframe_adserveragosto = pd.DataFrame(worksheet31.get_all_records())

#dataframe_adserveragosto['Plataforma'] = 'Agosto'
#dataframe_adserveragosto['Plataforma_geral'] = 'Agosto'

In [ ]:
# Inserir Planilha adserver_setembro
#link32 = 'https://docs.google.com/spreadsheets/d/1WuD3XWtOkH4vQ8ufrUBgIArECBQlAUmaNJ1C2FMWl-U/edit?usp=sharing'
#spreadsheet = gc.open_by_url(link32)

# Supondo que a guia se chama "Adserver_setembro"
#worksheet32 = spreadsheet.worksheet('Adserver_setembro')

# Obter os dados da planilha em um DataFrame
#dataframe_adserversetembro = pd.DataFrame(worksheet32.get_all_records())

# Adicionar as colunas 'Plataforma' e 'Plataforma_geral'
#dataframe_adserversetembro['Plataforma'] = 'Setembro'
#dataframe_adserversetembro['Plataforma_geral'] = 'Setembro'


In [ ]:
# Juntar os DataFrames
#merged_dataframe = pd.concat([dataframe_adserverjulho, dataframe_adserveragosto, dataframe_adserversetembro])

# Abra a planilha específica pelo URL
#planilha_link = 'https://docs.google.com/spreadsheets/d/1WuD3XWtOkH4vQ8ufrUBgIArECBQlAUmaNJ1C2FMWl-U/edit?usp=sharing'
#planilha = gc.open_by_url(planilha_link)

# Selecione a guia específica onde deseja salvar os dados (substitua 'Nome da Guia' pelo nome correto)
#guia = planilha.worksheet('Adserver')

# Limpe os dados existentes na guia (opcional)
#guia.clear()

# Escreva o DataFrame na guia
#set_with_dataframe(guia, merged_dataframe)

#print("Dados enviados com sucesso para a planilha no Google Sheets.")

In [ ]:
# Abrir a planilha pelo URL
spreadsheet = gc.open_by_url(link)

# URL e aba de origem

worksheet_ranking = gc.open_by_url(link).worksheet('Planilha Unificada')

# Carregar dados em DataFrame

dataframe_ranking= pd.DataFrame(worksheet_ranking.get_all_records())

# Função para verificar ou criar uma aba no Google Sheets
def get_or_create_worksheet(gc, link, sheet_name, rows=100, cols=20):
    try:
        return gc.open_by_url(link).worksheet(sheet_name)
    except gspread.exceptions.WorksheetNotFound:
        return gc.open_by_url(link).add_worksheet(title=sheet_name, rows=rows, cols=cols)

dataframe_ranking.columns = dataframe_ranking.columns.str.strip()

dataframe_ranking["Custo"] = dataframe_ranking["Custo"].astype(str)

dataframe_ranking["Custo"] = dataframe_ranking["Custo"].astype(str)

# Remover "R$" e espaços, se necessário
dataframe_ranking["Custo"] = dataframe_ranking["Custo"].replace('[R$\s]', '', regex=True)

# Remover os pontos que separam milhares
dataframe_ranking["Custo"] = dataframe_ranking["Custo"].str.replace(r'\.', '', regex=True)

# Substituir a vírgula pelo ponto decimal
dataframe_ranking["Custo"] = dataframe_ranking["Custo"].str.replace(',', '.', regex=False)

# Converter para numérico
dataframe_ranking["Custo"] = pd.to_numeric(dataframe_ranking["Custo"], errors="coerce").fillna(0)


# Limpeza das colunas que precisam ser numéricas
dataframe_ranking["Impressões"] = pd.to_numeric(dataframe_ranking["Impressões"], errors="coerce").fillna(0)
dataframe_ranking["Link clicks"] = pd.to_numeric(dataframe_ranking["Link clicks"], errors="coerce").fillna(0)
dataframe_ranking["Video views"] = pd.to_numeric(dataframe_ranking["Video views"], errors="coerce").fillna(0)
dataframe_ranking["Video views 100%"] = pd.to_numeric(dataframe_ranking["Video views 100%"], errors="coerce").fillna(0)

# Conversão para numérico
dataframe_ranking["Custo"] = pd.to_numeric(dataframe_ranking["Custo"], errors="coerce").fillna(0)

# Certifique-se de limpar e converter as colunas para numérico
colunas_para_converter = ["Custo", "Impressões", "Link clicks", "Video views", "Video views 100%"]

for coluna in colunas_para_converter:
    dataframe_ranking[coluna] = pd.to_numeric(dataframe_ranking[coluna], errors="coerce").fillna(0)

# Prossiga com o agrupamento
agrupado_plataformas = dataframe_ranking.groupby(["Plataforma_geral", "Tipo de Compra"]).agg({
    "Impressões": "sum",
    "Link clicks": "sum",
    "Custo": "sum",
    "Video views 100%": "sum"
}).reset_index()

# Calcular CTR para cada grupo
agrupado_plataformas["CTR"] = (agrupado_plataformas["Link clicks"] / agrupado_plataformas["Impressões"]) * 100
agrupado_plataformas["CTR"] = agrupado_plataformas["CTR"].apply(lambda x: f"{x:.2f}%" if x > 0 else "0.00%")

# Calcular VTR (Video views 100% / Impressões) apenas para o tipo de compra "CPV"
agrupado_plataformas["VTR"] = agrupado_plataformas.apply(
    lambda row: (row["Video views 100%"] / row["Impressões"]) * 100 if row["Tipo de Compra"] == "CPV" and row["Impressões"] > 0 else 0,
    axis=1
)
agrupado_plataformas["VTR"] = agrupado_plataformas["VTR"].apply(lambda x: f"{x:.2f}%" if x > 0 else "0.00%")

# Ordenar os dados por "Plataforma_geral" e "Tipo de compra"
agrupado_plataformas = agrupado_plataformas.sort_values(["Plataforma_geral", "Tipo de Compra"], ascending=[True, True])

# Exportar para o Google Sheets
aba_plataformas = get_or_create_worksheet(gc, link, 'Ranking Tipo de Compra')
aba_plataformas.clear()
set_with_dataframe(aba_plataformas, agrupado_plataformas)

# Ordenar os dados por "Plataforma_geral" e "Tipo de compra"
agrupado_plataformas = agrupado_plataformas.sort_values(["Plataforma_geral", "Tipo de Compra"], ascending=[True, True])

# Exportar para o Google Sheets
aba_plataformas = get_or_create_worksheet(gc, link, 'Ranking Tipo de Compra')
aba_plataformas.clear()
set_with_dataframe(aba_plataformas, agrupado_plataformas)


# Converter colunas para numérico
colunas_numericas = ["Impressões", "Link clicks", "Custo", "Video views", "Video views 100%"]
for coluna in colunas_numericas:
    dataframe_ranking[coluna] = pd.to_numeric(dataframe_ranking[coluna], errors="coerce").fillna(0)


<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2311/1055797130.py:26: SyntaxWarning: invalid escape sequence '\s'
  dataframe_ranking["Custo"] = dataframe_ranking["Custo"].replace('[R$\s]', '', regex=True)


In [ ]:
# Abrir a planilha pelo URL
spreadsheet = gc.open_by_url(link)

# Buscar a aba "Planilha Unificada" e carregar os dados em um DataFrame (apenas uma vez)
worksheet_unificada = spreadsheet.worksheet('Planilha Unificada')
dataframe_criativos = pd.DataFrame(worksheet_unificada.get_all_records())

# Função para verificar ou criar uma aba no Google Sheets
def get_or_create_worksheet(gc, link, sheet_name, rows=100, cols=20):
    try:
        return gc.open_by_url(link).worksheet(sheet_name)
    except gspread.exceptions.WorksheetNotFound:
        return gc.open_by_url(link).add_worksheet(title=sheet_name, rows=rows, cols=cols)

# Remover espaços extras nos nomes das colunas
dataframe_criativos.columns = dataframe_criativos.columns.str.strip()

# Limpar e converter a coluna "Custo" (caso tenha formatação monetária)
if "Custo" in dataframe_criativos.columns:
    dataframe_criativos["Custo"] = (
        dataframe_criativos["Custo"]
        .astype(str)
        .str.replace('[R$\s]', '', regex=True)   # Remove "R$" e espaços
        .str.replace(r'\.', '', regex=True)         # Remove pontos dos milhares
        .str.replace(',', '.', regex=False)         # Substitui vírgula decimal por ponto
    )

# Converter todas as colunas que precisam ser numéricas (evita repetições)
colunas_numericas = ["Impressões", "Link clicks", "Custo", "Video views", "Video views 100%"]
for coluna in colunas_numericas:
    dataframe_criativos[coluna] = pd.to_numeric(dataframe_criativos[coluna], errors="coerce").fillna(0)

# **Ranking de Veículos e Ad Sets**
agrupado_criativos = dataframe_criativos.groupby(
    ["Plataforma_geral", "Ad set name", "Tipo de Compra"]
).agg({
    "Impressões": "sum",
    "Link clicks": "sum",
    "Custo": "sum",
    "Video views": "sum",
    "Video views 100%": "sum"
}).reset_index()

# Calcular CTR
agrupado_criativos["CTR"] = (agrupado_criativos["Link clicks"] / agrupado_criativos["Impressões"]) * 100
agrupado_criativos["CTR"] = agrupado_criativos["CTR"].apply(lambda x: f"{x:.2f}%" if x > 0 else "0.00%")

# Calcular VTR (Video views 100% / Impressões) apenas para o tipo de compra "CPV"
agrupado_criativos["VTR"] = agrupado_criativos.apply(
    lambda row: (row["Video views 100%"] / row["Impressões"]) * 100 if row["Tipo de Compra"] == "CPV" and row["Impressões"] > 0 else 0,
    axis=1
)
agrupado_criativos["VTR"] = agrupado_criativos["VTR"].apply(lambda x: f"{x:.2f}%" if x > 0 else "0.00%")

# Ordenar e exportar Ranking de Veículos e Ad Sets
ranking_veiculos = agrupado_criativos.sort_values("Impressões", ascending=False).reset_index(drop=True)
aba_veiculos = get_or_create_worksheet(gc, link, 'Ranking Ad Sets')
aba_veiculos.clear()
set_with_dataframe(aba_veiculos, ranking_veiculos)

<>:23: SyntaxWarning: invalid escape sequence '\s'
<>:23: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2311/2248760217.py:23: SyntaxWarning: invalid escape sequence '\s'
  .str.replace('[R$\s]', '', regex=True)   # Remove "R$" e espaços
